In [ ]:
# Load environment variables (ANTHROPIC_API_KEY) from .env
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic import Anthropic, Omit, omit
from anthropic._types import SequenceNotStr
from anthropic.types import MessageParam

client = Anthropic()

In [ ]:
# Message helpers as before; chat() now also takes stop_sequences.

def add_user_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="user", content=text))

def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="assistant", content=text))

def chat(
    messages: list[MessageParam],
    *,
    model: str = "claude-sonnet-4-5",
    max_tokens: int = 1000,
    system: str | Omit = omit,
    temperature: float = 1.0,
    stop_sequences: SequenceNotStr[str] | Omit = omit) -> str:
    """Send the message history and return the reply.

    model: which Claude model to use; trades off capability, speed, and cost.
    max_tokens: hard cap on generated tokens — a limit, not a target (truncates).
    system: top-level instruction setting the model's role, rules, and tone;
        passed separately from messages. Defaults to `omit` (field dropped).
    temperature: sampling randomness, 0.0 (focused) to 1.0 (varied).
    stop_sequences: strings that halt generation the moment the model produces
        one (the sequence itself is not included in the output). Paired with an
        assistant prefill like "```json", this brackets the reply so you get
        back just the content between the fences - a simple way to coax clean,
        directly-parseable structured output.
    """
    response = client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        system=system,
        temperature=temperature,
        stop_sequences=stop_sequences
    )

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
# Structured output via prefill + stop.
messages: list[MessageParam] = []

add_user_message(messages, "Generate a very short event bridge rule as json")

# 1) prefill the assistant turn with "```json" so the model continues inside a JSON block
add_assistant_message(messages, "```json")

# 2) stop at the closing "```" so the reply is pure JSON, ready to parse
answer = chat(messages, stop_sequences=["```"])

print(answer)